In [4]:
import os
import glob
import numpy as np
from skimage.segmentation import felzenszwalb, mark_boundaries
from skimage.util import img_as_float
from skimage import io, color
from skimage.measure import label

def generate_pseudolabels_felzenszwalb(image_path, output_dir, vis_dir):
    """
    為單張2D彩色圖片生成superpixel pseudolabels (使用Felzenszwalb演算法)。

    Args:
        image_path (str): 輸入圖片的路徑。
        output_dir (str): 儲存pseudolabel的路徑。
        vis_dir (str): 儲存視覺化結果的路徑。
    """
    # --- 1. 參數設定 (這是您需要主要調整的部分) ---
    
    # 控制超級像素的尺寸，值越大，超級像素越大。建議從 100 到 1000 之間嘗試。
    SCALE = 500
    
    # 分割前的預平滑，有助於減少噪點影響。
    SIGMA = 0.8
    
    # 合併後處理，確保超級像素不小於這個尺寸。
    MIN_SIZE = 100

    # --- 讀取和預處理影像 ---
    try:
        image = io.imread(image_path)
    except Exception as e:
        print(f"Error reading {image_path}: {e}")
        return

    # 確保圖片是RGB格式 (移除可能存在的Alpha通道)
    if image.shape[-1] == 4:
        image = image[..., :3]
        
    image_float = img_as_float(image)

    # --- 2. 使用 Felzenszwalb 演算法生成 Superpixels ---
    # `segments` 是一個 2D 陣列，每個像素的值代表其所屬的 superpixel ID。
    # multichannel=True 是必須的，因為我們處理的是彩色圖片。
    segments = felzenszwalb(image_float, 
                           scale=SCALE, 
                           sigma=SIGMA, 
                           min_size=MIN_SIZE)
    
    # Felzenszwalb的標籤從0開始，我們將它+1，使其從1開始，避免0被誤認為背景。
    segments = segments + 1

    # --- 3. (可選) 移除背景區域的超級像素 ---
    # 這裡我們用一個簡單的亮度閾值來定義前景。
    # 例如，如果內視鏡影像周圍有黑色邊框，我們可以移除這些區域的標籤。
    # 轉換到灰度圖以計算亮度
    gray_image = color.rgb2gray(image_float)
    # 假設亮度低於0.1的區域是背景
    foreground_mask = gray_image > 0.1 
    
    # 將背景區域的標籤設為0
    segments_masked = segments * foreground_mask

    # --- 4. 重新標記，確保標籤是連續的 (1, 2, 3, ...) ---
    # 因為移除背景後，標籤可能不再連續。
    # label() 函式會找到所有連通區域並重新賦予從1開始的連續標籤。
    final_labels, num_labels = label(segments_masked, background=0, return_num=True)
    
    # 取得檔案基本名稱，用於儲存
    base_name = os.path.splitext(os.path.basename(image_path))[0]
    
    # --- 5. 儲存 Pseudolabel 結果 ---
    # 儲存為16位元的PNG以支援超過255個標籤
    output_filename = os.path.join(output_dir, f"{base_name}_label.png")
    io.imsave(output_filename, final_labels.astype(np.uint16))

    # --- 6. 產生並儲存視覺化結果 ---
    vis_image = mark_boundaries(image_float, final_labels)
    vis_filename = os.path.join(vis_dir, f"{base_name}_vis.png")
    io.imsave(vis_filename, (vis_image * 255).astype(np.uint8))
    
    print(f"Processed {base_name} successfully, generated {num_labels} superpixels.")


def main():
    # --- 設定您的路徑 ---
    INPUT_DIR = '/home/U116med/data/polyp/TrainDataset/images' 
    OUTPUT_DIR = './polypdata/pseudolabels_felzenszwalb/' # 建議用新資料夾以區分
    VISUALIZATION_DIR = './polypdata/visualizations_felzenszwalb/'

    # 建立輸出資料夾
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    os.makedirs(VISUALIZATION_DIR, exist_ok=True)

    image_paths = glob.glob(os.path.join(INPUT_DIR, '*.png'))
    image_paths.extend(glob.glob(os.path.join(INPUT_DIR, '*.jpg'))) # 支援jpg
    
    if not image_paths:
        print(f"Error: No images found in {INPUT_DIR}. Please check the path.")
        return

    print(f"Found {len(image_paths)} images. Starting processing with Felzenszwalb...")
    
    for path in image_paths:
        generate_pseudolabels_felzenszwalb(path, OUTPUT_DIR, VISUALIZATION_DIR)
        
    print("All images processed.")


if __name__ == '__main__':
    main()

Found 1450 images. Starting processing with Felzenszwalb...
Processed 1 successfully, generated 45 superpixels.


/tmp/ipykernel_521040/3679557647.py:75: UserWarning: ./polypdata/pseudolabels_felzenszwalb/1_label.png is a low contrast image
  io.imsave(output_filename, final_labels.astype(np.uint16))
/tmp/ipykernel_521040/3679557647.py:75: UserWarning: ./polypdata/pseudolabels_felzenszwalb/10_label.png is a low contrast image
  io.imsave(output_filename, final_labels.astype(np.uint16))


Processed 10 successfully, generated 53 superpixels.
Processed 101 successfully, generated 64 superpixels.


/tmp/ipykernel_521040/3679557647.py:75: UserWarning: ./polypdata/pseudolabels_felzenszwalb/101_label.png is a low contrast image
  io.imsave(output_filename, final_labels.astype(np.uint16))
/tmp/ipykernel_521040/3679557647.py:75: UserWarning: ./polypdata/pseudolabels_felzenszwalb/102_label.png is a low contrast image
  io.imsave(output_filename, final_labels.astype(np.uint16))


Processed 102 successfully, generated 45 superpixels.
Processed 103 successfully, generated 53 superpixels.


...

/tmp/ipykernel_521040/3679557647.py:75: UserWarning: ./polypdata/pseudolabels_felzenszwalb/ck2bxskgxxzfv08386xkqtqdy_label.png is a low contrast image
  io.imsave(output_filename, final_labels.astype(np.uint16))


Processed ck2bxskgxxzfv08386xkqtqdy successfully, generated 84 superpixels.


/tmp/ipykernel_521040/3679557647.py:75: UserWarning: ./polypdata/pseudolabels_felzenszwalb/ck2bxw18mmz1k0725litqq2mc_label.png is a low contrast image
  io.imsave(output_filename, final_labels.astype(np.uint16))


Processed ck2bxw18mmz1k0725litqq2mc successfully, generated 124 superpixels.
Processed ck2da7fwcjfis07218r1rvm95 successfully, generated 213 superpixels.
All images processed.


/tmp/ipykernel_521040/3679557647.py:75: UserWarning: ./polypdata/pseudolabels_felzenszwalb/ck2da7fwcjfis07218r1rvm95_label.png is a low contrast image
  io.imsave(output_filename, final_labels.astype(np.uint16))
